# Compositional (indirect) noise and the compact choked nozzle

A composition inhomogeneity: a pocket of gas with a different mixture, hence a different $\gamma$ and $R$: radiates sound when it is accelerated through a nozzle, exactly as an entropy (temperature) spot does.
This is **compositional**, or **indirect**, combustion noise (Magri, *JFM* 2016).

A choked nozzle can be closed three ways in Nefes, and they are *not* equivalent for this mechanism:

1. **The hand-written compact closure** `PerturbationBC.choked_nozzle()`: the textbook Marble–Candel formula $g = R\,f + R_s\,h$, a 3-wave $(f,g,h)$ relation.
2. **The inherited compact element** `choked_nozzle_outlet(A*)`: the critical-mass-flux jump, linearized **by complex step** through its *full* state dependence.
3. **A resolved nozzle**: area-change elements and ducts, with the near-throat remainder lumped.

For the **acoustic** reflection $R$ the three agree in the compact limit.
But the hand-written closure carries only $(f,g,h)$: it has **no composition column** $R_\xi$: so it silently drops compositional noise.
Routes 2 and 3 linearize the *actual* jump, so the same complex step that gives $R$ and the entropy coupling $R_s$ also gives $R_\xi$ **for free**.

This notebook validates the acoustic limit, demonstrates the compositional noise the inherited element captures, and shows the limitation of the hand-written closure.
A closing note places it against the $M=1$ subsonic scope.

In [ ]:
import warnings

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.elements import catalog as cat
from nefes.perturbation import PerturbationBC, CompositionalNoiseWarning
from nefes.perturbation import build_acoustic_blocks, assemble_acoustic, edge_caloric, char_to_dx
from nefes.thermo import SpeciesDatabase, EQ_FROZEN
from nefes.plotting import use_nefes_theme, COLORWAY

use_nefes_theme()
R_AIR, GAMMA = 287.0, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)
GM1 = GAMMA - 1.0

## 1. Inert acoustic limit: the three terminations agree

A high-pressure reservoir discharges through a feed duct to a choked nozzle (perfect-gas air).
We drive a unit acoustic wave at the inlet and read the reflection $g/f$ at the **application plane** (the duct head, just upstream of the nozzle).

* the **compact** termination is the inherited `choked_nozzle_outlet(A*)`;
* the **resolved** termination resolves part of the contraction (an `isentropic_area_change` and a throat duct) and lumps only the near-throat remainder.

Both stay strictly subsonic on the application plane: the sonic point lives *inside* the lumped throat.

In [ ]:
PT, TT, A_IN, A_STAR = 1.6e5, 300.0, 0.05, 0.03
FREQ = np.linspace(1.0, 1500.0, 300)  # Hz

DRIVE_ACOUSTIC = PerturbationBC.anechoic(driven=("acoustic",))  # unit acoustic wave in, absorb returning sound
DRIVE_ENTROPY = PerturbationBC.anechoic(driven=("entropy",))  # unit entropy wave in, absorb returning sound


def build_compact(inlet, analytic=False):
    """Feed duct -> compact choked nozzle (inherited, or the hand-written analytic closure)."""
    nb = PerturbationBC.choked_nozzle() if analytic else None
    nodes = [
        cat.total_pressure_inlet(PT, TT, perturbation_bc=inlet),
        cat.duct(0.6),
        cat.choked_nozzle_outlet(A_STAR, perturbation_bc=nb),
    ]
    net = nefes.Network(nefes.perfect_gas(R_AIR, GAMMA), nodes=nodes, edges=[(0, 1, A_IN), (1, 2, A_IN)])
    sol = net.solve()
    assert sol.converged, (sol.residual_norm, sol.print_residuals())
    return sol


def build_resolved(inlet, A_mid=0.04):
    """Feed duct -> isentropic contraction -> throat duct -> lumped choked throat."""
    nodes = [
        cat.total_pressure_inlet(PT, TT, perturbation_bc=inlet),
        cat.duct(0.6),
        cat.isentropic_area_change(),
        cat.duct(0.25),
        cat.choked_nozzle_outlet(A_STAR),
    ]
    edges = [(0, 1, A_IN), (1, 2, A_IN), (2, 3, A_mid), (3, 4, A_mid)]
    net = nefes.Network(nefes.perfect_gas(R_AIR, GAMMA), nodes=nodes, edges=edges)
    sol = net.solve()
    assert sol.converged, (sol.residual_norm, sol.print_residuals())
    return sol


def reflection(sol, plane=1):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return sol.forced_response(FREQ).reflection_at(plane)


sol_c = build_compact(DRIVE_ACOUSTIC)
sol_a = build_compact(DRIVE_ACOUSTIC, analytic=True)
sol_r = build_resolved(DRIVE_ACOUSTIC)
R_compact = reflection(sol_c)
R_analytic = reflection(sol_a)
R_resolved = reflection(sol_r)

M_plane = sol_c.edge(1)["M"]
R_mc = (2.0 - GM1 * M_plane) / (2.0 + GM1 * M_plane)
print(f"application-plane Mach M = {M_plane:.3f}")
print(f"Marble-Candel R(M)       = {R_mc:.4f}")
print(
    f"inherited vs analytic closure  max|dR| = {np.max(np.abs(R_compact - R_analytic)):.2e}  (identical to round-off)"
)

In [ ]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=(r"$|R(f)|$", r"$\angle R(f)\;[\mathrm{deg}]$"), horizontal_spacing=0.11
)
for y, name, col in [
    (np.abs(R_compact), "compact (inherited)", COLORWAY[0]),
    (np.abs(R_resolved), "resolved", COLORWAY[2]),
]:
    fig.add_trace(go.Scatter(x=FREQ, y=y, name=name, line=dict(color=col)), row=1, col=1)
fig.add_hline(
    y=R_mc,
    line=dict(dash="dash", color="#52606d"),
    annotation_text="Marble-Candel R(M)",
    annotation_position="bottom right",
    row=1,
    col=1,
)
for y, name, col in [
    (np.degrees(np.angle(R_compact)), "compact (inherited)", COLORWAY[0]),
    (np.degrees(np.angle(R_resolved)), "resolved", COLORWAY[2]),
]:
    fig.add_trace(go.Scatter(x=FREQ, y=y, name=name, line=dict(color=col), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text=r"$f\;[\mathrm{Hz}]$", row=1, col=1)
fig.update_xaxes(title_text=r"$f\;[\mathrm{Hz}]$", row=1, col=2)
fig.update_layout(height=420, width=960, title_text="Compact vs resolved choked nozzle (inert)")
fig.show()

The inherited compact element and the hand-written analytic closure give the **same** $R$ to round-off: the complex-step linearization of the critical-mass-flux jump *is* the Marble–Candel formula, no hand-coded coefficient needed.

The compact $|R|$ is **flat** at $R(M)$: a compact nozzle's reflection depends only on the application-plane Mach.
The resolved nozzle coincides with it as $f\to 0$ (the section is acoustically compact there) and departs at higher frequency, where the finite length of the resolved contraction adds its own propagation phase.
Compactness is precisely the **low-frequency** limit of the resolved nozzle.

### Entropy noise: the same comparison

The reflection $R$ is only one entry of the nozzle's scattering.
The off-diagonal $R_s$: an arriving **entropy** (temperature) spot radiating sound as it accelerates: is *indirect (entropy) noise*, and it compares compact-vs-resolved in exactly the same spirit.
We drive a unit **entropy** wave at the (anechoic) inlet and read the sound it radiates back upstream, $g$, at the application plane.

(The same comparison against the De Domenico *et al.* (2019) compact model: acoustic **and** entropic transfer functions of a lossy nozzle: is the validation in `entropy_generator.ipynb` and `tests/test_perturbation_dedomenico.py`.)

In [ ]:
def upstream_sound(sol, plane=1):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return sol.forced_response(FREQ).waves(plane)[:, 1]  # g: the upstream-radiated wave


sol_ce = build_compact(DRIVE_ENTROPY)
sol_re = build_resolved(DRIVE_ENTROPY)
g_compact = upstream_sound(sol_ce)
g_resolved = upstream_sound(sol_re)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(r"$|g(f)|$  (entropy noise)", r"$\angle g(f)\;[\mathrm{deg}]$"),
    horizontal_spacing=0.11,
)
for y, name, col in [
    (np.abs(g_compact), "compact (inherited)", COLORWAY[0]),
    (np.abs(g_resolved), "resolved", COLORWAY[2]),
]:
    fig.add_trace(go.Scatter(x=FREQ, y=y, name=name, line=dict(color=col)), row=1, col=1)
for y, col in [(np.degrees(np.angle(g_compact)), COLORWAY[0]), (np.degrees(np.angle(g_resolved)), COLORWAY[2])]:
    fig.add_trace(go.Scatter(x=FREQ, y=y, line=dict(color=col), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text=r"$f\;[\mathrm{Hz}]$", row=1, col=1)
fig.update_xaxes(title_text=r"$f\;[\mathrm{Hz}]$", row=1, col=2)
fig.update_layout(height=420, width=960, title_text="Entropy noise: compact vs resolved choked nozzle (inert)")
fig.show()
print(f"compact  |g|: {np.abs(g_compact).min():.2f}..{np.abs(g_compact).max():.2f}  (flat -> constant R_s)")
print(f"resolved |g|: {np.abs(g_resolved).min():.2f}..{np.abs(g_resolved).max():.2f}")

Same verdict, with one extra twist worth noting.
The **magnitude** story is identical to the reflection: compact $|g|$ is **flat** (a single $R_s$), and the resolved nozzle meets it at $f\to 0$ then departs.
But the **phase** winds far faster than the reflection's, because the entropy spot rides the slow **convective** speed $u$ across the nozzle (a delay $L/u$), not the acoustic $L/(c\pm u)$: so the resolved entropy noise becomes non-compact at a *lower* frequency than the reflection does.
That convective phase is the heart of the non-compact entropy-noise corrections (De Domenico; Duran–Moreau).

## 2. Compositional noise: what the inherited linearization captures

Now make the flow **reacting and multi-stream**.
We declare two feed streams up front, air and hydrogen, and feed a single **premixed inlet** that blends them at a 20:1 air-to-H2 mass ratio.
A composition can only fluctuate if the gas is a blend of separately-tracked constituents; declaring air and hydrogen as distinct streams gives the one premixed inlet exactly that, so its mixture fraction is a live degree of freedom, with no dedicated fuel inlet and no in-line injector.
The mixture is kept frozen (no flame element forces equilibrium), so a fluctuation is a pure *composition* spot that convects to the choked nozzle.

![Network topology](compositional_noise_topology.png)

Each declared stream carries a **mixture fraction** $\xi$: a scalar convected at the mean flow speed that marks how much of the local gas came from that stream (air versus hydrogen).
A fluctuation $\xi'$ is a *composition wave*: shifting mass from one stream to the other changes the local $\gamma$ and $R$, and therefore the critical mass flux at the nozzle.

We read the nozzle's outlet row in characteristic coordinates,

$$ g = R\,f + R_s\,h + R_\xi\cdot\boldsymbol{\xi}, $$

extracting $R$ (acoustic reflection), $R_s$ (entropy noise) and $R_\xi$ (compositional noise) directly from the linearized row.

In [ ]:
AIR = {"O2": 0.21, "N2": 0.79}
lib = SpeciesDatabase().select(["O2", "N2", "H2", "H2O", "OH", "H", "O"])
# Declare the two feed streams up front: air and hydrogen.  A single premixed inlet then blends
# them (mode="declared"), so the mixture fraction is a live degree of freedom -- there is no
# dedicated fuel inlet and no in-line injector.
gas = nefes.equilibrium(lib, streams={"air": AIR, "H2": {"H2": 1.0}}, mode="declared")


def reacting_nozzle(nozzle_bc=None, inlet_bc=None):
    """Premixed (air + H2) inlet -> feed duct -> approach duct -> compact choked nozzle.

    The premix carries 0.42 kg/s at a 20:1 air-to-H2 mass ratio, kept frozen (no flame), so a
    driven mixture-fraction fluctuation is a pure composition spot that convects to the nozzle.
    """
    nodes = [
        cat.mass_flow_inlet(
            0.42,
            300.0,
            composition={"air": 20.0, "H2": 1.0},
            basis="mass",
            perturbation_bc=inlet_bc,
            name="premixed inlet",
        ),
        cat.duct(0.4, name="feed duct"),
        cat.duct(0.5, name="nozzle approach"),
        cat.choked_nozzle_outlet(0.012, perturbation_bc=nozzle_bc, name="choked nozzle"),
    ]
    edges = [(0, 1, 0.02), (1, 2, 0.02), (2, 3, 0.02)]
    return nefes.Network(gas, nodes=nodes, edges=edges, edge_models=[EQ_FROZEN] * 3)


def nozzle_row(sol, stamped):
    """(M, R, R_s, R_xi) of the choked-nozzle outlet row in (f, g, h, xi) coordinates."""
    prob, x = sol.problem, sol.x
    e = 2  # the choked-nozzle outlet edge
    cal = edge_caloric(prob, x)[e]
    ns, r0 = int(prob.n_solve), int(prob.node_row_ptr[3])  # node 3 is the nozzle
    if stamped:  # an explicit closure overwrites the row
        A = assemble_acoustic(0.0, build_acoustic_blocks(prob, x), with_boundaries=True).tocsr()
        row = np.asarray(A[r0, ns * e : ns * e + ns].todense()).ravel()
    else:  # the inherited J_alg row
        row = np.asarray(build_acoustic_blocks(prob, x).J_alg[r0, ns * e : ns * e + ns].todense()).ravel()
    st = sol.edge(e)
    cf, cg, ch = row[:3] @ char_to_dx(st["rho"], st["c"], st["u"], st["area"], cal)
    return st["M"], -cf / cg, -ch / cg, -row[3:] / cg


sol_in = reacting_nozzle().solve()
assert sol_in.converged, (sol_in.residual_norm, sol_in.print_residuals())
streams = list(sol_in.mixture_fractions(2))
M, R, R_s, R_xi = nozzle_row(sol_in, stamped=False)
print("declared feed streams (mixture fractions):", streams)
print(f"nozzle approach Mach M = {M:.3f}")
print(f"R   (acoustic)      = {R.real:+.4f}    [Marble-Candel {(2 - GM1 * M) / (2 + GM1 * M):.4f}]")
print(f"R_s (entropy noise) = {R_s.real:+.4g}")
for nm, v in zip(streams, R_xi):
    print(f"R_xi[{nm!r}] (compositional noise) = {v.real:+.2f}")

The inherited outlet row recovers the Marble–Candel **acoustic** reflection $R$, and it also carries a **non-zero** compositional column $R_\xi$.
The hydrogen stream dominates that column: its mixture fraction moves $\gamma$ and $R$ the most.
That column *is* the compositional-noise coupling; it comes from the same complex-step assembly as $R$ and $R_s$.

**What we drive, and what a composition wave is.**
To see that column radiate, we drive a single wave at the premixed inlet and watch the sound come out.
The `anechoic(driven=("H2",))` boundary imposes one incoming wave: a fluctuation of the mixture that **raises the hydrogen fraction and lowers the air fraction by the same amount**, at constant total mass.
No acoustic wave is driven ($f = 0$) and the returning sound is absorbed, so the composition path is probed on its own.

A **composition wave** is an oscillation in *what the gas is made of*, here the local fraction of hydrogen-origin to air-origin mass, carried downstream at the mean flow speed $u$: the same convective ride an entropy (temperature) spot takes.
It shifts the local $\gamma$ and $R$, and so the density at fixed pressure, but on its own it carries **no pressure and no mass-flow fluctuation**: it is acoustically *silent* while it convects along a uniform duct.
This is exactly the **equivalence-ratio wave** of premixed-combustion theory: the total mass flow is held constant and only the mixture (the equivalence ratio) oscillates, at fixed pressure, riding to the nozzle at $u$.
Driving the stream fraction directly keeps the mass flow **exactly** constant: raising one stream lowers the other, so there is no added-mass fluctuation to contaminate the composition wave.

How is such a wave made physically?
By modulating the fuel feed of a premixed system: an acoustic velocity fluctuation upstream re-proportions the mixture (the mechanism `equivalence_ratio_instability.ipynb` builds).
The wave turns into sound only where the flow is **accelerated** or otherwise non-uniform, above all at the choked nozzle, and that radiated sound is the compositional noise.

In [ ]:
fuel = streams[-1]  # the hydrogen stream's mixture fraction (the composition we perturb)
# drive a unit equivalence-ratio wave at the premixed inlet: raise H2, lower air by the same
# amount, at constant total mass; no acoustic wave in, returning sound absorbed.
sol_d = reacting_nozzle(inlet_bc=PerturbationBC.anechoic(driven=(fuel,))).solve()
assert sol_d.converged, (sol_d.residual_norm, sol_d.print_residuals())
fr = sol_d.forced_response(FREQ)
i_xi = fr.wave_labels.index(fuel)

xi_in = fr.waves(0)[:, i_xi]  # the driven composition wave at the inlet
xi_noz = fr.waves(2)[:, i_xi]  # the same wave arrived at the nozzle (convected at u)
f_duct = fr.waves(1)[:, 0]  # acoustic f in the duct: silent (~0) all the way to the nozzle
g_noz = fr.waves(2)[:, 1]  # upstream-radiated wave g at the nozzle: the compositional noise

fig = go.Figure()
fig.add_trace(go.Scatter(x=FREQ, y=np.abs(xi_in), name=r"$|\xi'|$ driven at inlet", line=dict(color=COLORWAY[3])))
fig.add_trace(
    go.Scatter(x=FREQ, y=np.abs(xi_noz), name=r"$|\xi'|$ arrived at nozzle", line=dict(color=COLORWAY[3], dash="dot"))
)
fig.add_trace(go.Scatter(x=FREQ, y=np.abs(g_noz), name=r"$|g|$ radiated at nozzle", line=dict(color=COLORWAY[0])))
fig.add_trace(
    go.Scatter(x=FREQ, y=np.abs(f_duct), name=r"$|f|$ in the duct ($\approx 0$)", line=dict(color=COLORWAY[1]))
)
fig.update_layout(
    height=400,
    width=820,
    xaxis_title=r"$f\;[\mathrm{Hz}]$",
    title="A silent composition wave convects to the nozzle and radiates sound (compositional noise)",
)
fig.show()
print(f"unit composition wave in -> |g| radiated at nozzle = {np.abs(g_noz).min():.1f} .. {np.abs(g_noz).max():.1f}")
print(f"acoustic |f| in the duct (the wave is silent): max = {np.abs(f_duct).max():.2e}")

**Reading the plot.**
Follow the driven composition wave from the inlet to the nozzle.

* The mixture fraction $\xi'$ is driven at unit amplitude at the inlet and **convects unchanged** to the nozzle: $|\xi'| = 1$ all the way (the solid and dotted traces coincide), riding the mean flow at speed $u$ as a pure phase delay, with no growth or decay.
* Along the whole duct the acoustic $f$ is **zero**: the composition wave carries no pressure or mass-flow fluctuation, so a uniform duct keeps it silent. There is no injector or area change upstream to scatter it.
* At the **choked nozzle** the arriving composition wave is accelerated and radiates the upstream-running wave $g$: the compositional noise. This is produced entirely by the inherited linearization, with no hand-written scalar closure.

Because the inlet is non-reflecting, that radiated $g$ is absorbed there and never returns, so $f$ stays zero everywhere upstream: the plot isolates the nozzle as the sole generator of sound from a silent, convected composition wave.
This is the indirect-noise analogue of the entropy-noise chain, driven by composition instead of temperature.

## 3. The limitation: the hand-written compact closure drops it

Swap the inherited element for the hand-written `PerturbationBC.choked_nozzle()` closure and read the same row.
The closure now takes the **backend-correct** effective $\gamma = \rho c^2/p$ from the state, so its acoustic reflection $R$ matches the inherited value.
But it overwrites the row with a 3-wave $(f,g,h)$ relation: its composition column is **identically zero**.
The structural gap is the missing $R_\xi$: there is simply no slot for a composition fluctuation in the closed form.

In [ ]:
sol_an = reacting_nozzle(PerturbationBC.choked_nozzle()).solve()
assert sol_an.converged, (sol_an.residual_norm, sol_an.print_residuals())
Ma, Ra, R_sa, R_xia = nozzle_row(sol_an, stamped=True)
print(f"analytic closure:  R = {Ra.real:+.4f}   (inherited R = {R.real:+.4f}: matches: correct gamma)")
print(f"                   R_s = {R_sa.real:+.4g}")
for nm, v in zip(streams, R_xia):
    print(f"  R_xi[{nm!r}] = {v.real:+.2e}   (dropped)")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sol_an.forced_response(np.array([200.0]))
print("\nCompositionalNoiseWarning fired:", any(issubclass(w.category, CompositionalNoiseWarning) for w in caught))

In [ ]:
labels = [f"R_xi[{s}]" for s in streams]
fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=np.abs(R_xi), name="inherited element", marker_color=COLORWAY[0]))
fig.add_trace(go.Bar(x=labels, y=np.abs(R_xia), name="analytic closure", marker_color=COLORWAY[4]))
fig.update_layout(
    height=400,
    width=720,
    barmode="group",
    yaxis_title=r"$|R_\xi|$  (compositional-noise coupling)",
    title="Compositional noise: captured (inherited) vs dropped (analytic closure)",
)
fig.show()

The hand-written closure carries the **entropy** off-diagonal $R_s$ (entropy noise) but has no composition column at all.
That is exactly the case the `CompositionalNoiseWarning` flags: a compact analytic nozzle closure on a reacting flow.

**The takeaway:** for compositional noise at a compact nozzle, use the **inherited** `choked_nozzle_outlet` element, or **resolve** the nozzle.
Both linearize the real jump and get $R_\xi$ from the same complex step that gives $R$ and $R_s$: no new formula, no validation case of its own.

## 4. The $M = 1$ boundary

Everything here stays strictly **subsonic** on the application plane: Nefes v1 scope.
At $M=1$ the upstream acoustic characteristic has zero speed: its duct transit time $\tau_- = L/(c-u)\to\infty$, the boundary-value problem changes type, and the isentropic area–Mach relation turns over.
This is a genuine singularity of the sonic point, not a coding choice.

The **compact** closure is precisely the way around it at a boundary: it imposes the sonic condition *algebraically*, lumping the sonic point **outside** the domain so the application plane stays subsonic.
The **resolved** route works arbitrarily close to choke ($M\to 1^-$) but not at the point itself.
Propagating the acoustic field *through* a sonic throat needs a different solver: integrating the linearized-Euler equations through the $M(x)$ profile with the sonic-regularity (L'Hôpital) condition (Stow–Dowling / Duran–Moreau).
That distributed, through-throat nozzle response: the proper home for non-compact compositional noise: is deferred with the supersonic scope.

## Export for Nemo

The reacting compositional-noise mean flow is available as `sol_in` (a `Solution`), built on a **declared-stream** model (`equilibrium(streams=..., mode="declared")`).
Save it to a UI-readable YAML with `sol_in.to_yaml(path)` (topology, the declared streams and per-inlet blends, plus the embedded mean-flow fields), then open the file in Nemo.

In [ ]:
import os
import tempfile

_out = os.path.join(tempfile.mkdtemp(), "compositional_noise.yaml")
sol_in.to_yaml(_out)  # topology + declared streams + the embedded mean-flow results
print("saved case:", _out)